In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import os

from models.generator import Generator

def load_generator(model_path, device):
    model = Generator().to(device)
    checkpoint = torch.load(model_path, map_location=device)
    if 'generator_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['generator_state_dict'])
    else:
        model.load_state_dict(checkpoint)
    model.eval()
    return model

def deblur_image(generator, input_image_path, output_image_path, device):
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    
    inv_normalize = transforms.Normalize(
        mean=[-1, -1, -1],
        std=[2, 2, 2]
    )

    img = Image.open(input_image_path).convert('RGB')
    input_tensor = transform(img).unsqueeze(0).to(device)  # Shape: (1, 3, H, W)

    with torch.no_grad():
        output_tensor, _ = generator(input_tensor)
        output_tensor = output_tensor.squeeze(0).cpu()
        output_tensor = (output_tensor * 0.5) + 0.5
        output_img = transforms.ToPILImage()(output_tensor.clamp(0, 1))

    output_img.save(output_image_path)
    print(f"Saved deblurred image to: {output_image_path}")

if __name__ == "__main__":
    model_weights = "checkpoints/checkpoint_epoch_26.pth"  # Path for checkpoint or model file
    blurry_image_path = "D:/Image_Deblurring/Dataset/ValidationDataset/horse.png"       # Path to your blurry input image
    sharp_output_path = "D:/Image_Deblurring/Dataset/ValidationDataset/horse_output.png"  # Path where sharp image will be saved

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    generator = load_generator(model_weights, device)
    deblur_image(generator, blurry_image_path, sharp_output_path, device)